In [34]:
import torch
from torch.utils.data import Dataset, DataLoader
print(torch.backends.cuda.is_flash_attention_available())

False


In [35]:
import pandas as pd

df = pd.read_csv(
    "data/shakespeare/shakespeare.csv"
)

print("Rows:", len(df))
print("Character IDs:", df["character_id"].nunique())

Rows: 4226158
Character IDs: 1129


In [36]:
texts = []

for character_id, group in df.groupby(
    "character_id",
    sort=False
):

    # First window contains the beginning
    first_x = group.iloc[0]["x"]

    # Every following character comes from y
    ys = (
        group["y"]
        .fillna("")
        .astype(str)
    )

    text = (
        first_x
        + "".join(ys)
    )

    texts.append(text)


# Separate dialogue sequences
text = "\n\n".join(texts)


print("Total characters:", len(text))
print()
print(text[:1000])

Total characters: 4318734

You sad Andronici, have done with woes; Give sentence on the execrable wretch That hath been breeder of these dire events. 

Give us the proudest prisoner of the Goths, That we may hew his limbs, and on a pile Ad manes fratrum sacrifice his flesh Before this earthy prison of their bones, That so the shadows be not unappeas'd, Nor we disturb'd with prodigies on earth. Away with him, and make a fire straight; And with our swords, upon a pile of wood, Let's hew his limbs till they be clean consum'd. Exeunt TITUS' SONS, with ALARBUS See, lord and father, how we have perform'd Our Roman rites: Alarbus' limbs are lopp'd, And entrails feed the sacrificing fire, Whose smoke like incense doth perfume the sky. Remaineth nought but to inter our brethren, And with loud 'larums welcome them to Rome. Proud Saturnine, interrupter of the good That noble-minded Titus means to thee! And that he will and shall, if Lucius live. My lord, you are unjust, and more than so: In wrong

In [37]:
split = int(
    0.9 * len(text)
)

train_text = text[:split]
val_text = text[split:]


print(
    "Train characters:",
    len(train_text)
)

print(
    "Val characters:",
    len(val_text)
)

Train characters: 3886860
Val characters: 431874


In [38]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

def decode(tokens):
    return ''.join(itos[i] for i in tokens)
print(vocab_size)


80


In [39]:
train_data = torch.tensor([stoi[c] for c in train_text], dtype=torch.long)
val_data   = torch.tensor([stoi[c] for c in val_text], dtype=torch.long)

class ShakespeareDataset(Dataset):
    def __init__(self, data, seq_len) -> None:
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, index):
        x = self.data[index:index + self.seq_len] # the current letter
        y = self.data[index+ 1: index + self.seq_len + 1] # the next letter

        return x, y


seq_len = 512
batch_size = 64

train_dataset = ShakespeareDataset(train_data, seq_len)
val_dataset = ShakespeareDataset(val_data, seq_len)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [40]:
from torch import nn
from torch.nn import functional as F
import math

In [41]:
class RotaryEmbedding(nn.Module):
    def __init__(self,head_dim, max_seq_len = 4096, base = 10000) -> None:
        super().__init__()
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        self.base = base
        # Frequencies for each pair of dimensions.
        #
        # Shape:
        # (head_dim / 2)
        #
        inv_freq = 1.0 / (
            base ** (torch.arange(0,head_dim,2).float()/ head_dim)
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self,seq_len):
        device = self.inv_freq.device
        positions = torch.arange(
            seq_len,
            device = device,
            dtype=self.inv_freq.dtype
        )

        freqs = torch.outer(positions,self.inv_freq)

        # Duplicate frequencies so that:
        # [freq0, freq1, freq2]
        # becomes
        # [freq0, freq0, freq1, freq1, freq2, freq2]
        # matching pairs of dimensions.

        emb = torch.cat([freqs, freqs],dim=-1)

        cos = emb.cos()
        sin = emb.sin()

        # Shape:
        # (1, 1, seq_len, head_dim)

        self.register_buffer(
            "cos_cached",
            cos[None, None, :, :],
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            sin[None, None, :, :],
            persistent=False
        )

    def rotate_half(self,x):
        # Split:
        # [x0 x1 x2 x3]
        # into:
        # [x0 x1] [x2 x3]
        # and rotate each pair.

        x1 = x[..., :x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]

        return torch.cat(
            (-x2, x1),
            dim=-1
        )
    
    def forward(self, x, positions):
        # x:
        # (B, heads, T, head_dim)
        # positions:
        # (T,)

        max_position = positions.max().item()

        if max_position >= self.cos_cached.shape[2]:
            self._build_cache(
                max_position + 1
            )

        cos = self.cos_cached[:,:,positions,:]
        sin = self.sin_cached[:,:,positions,:]

        return (
            x * cos
            + self.rotate_half(x) * sin
        )

In [42]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd,n_head,n_kv_head,dropout=0.0) -> None:
        super().__init__()

        assert n_head % n_kv_head == 0
        assert n_embd % n_head == 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head

        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embed,n_head * self.head_dim,bias=False)
        self.k_proj = nn.Linear(n_embed,n_kv_head * self.head_dim,bias=False)
        self.v_proj = nn.Linear(n_embed,n_kv_head * self.head_dim,bias=False)

        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout_p = dropout

        self.rope = RotaryEmbedding(head_dim=self.head_dim,max_seq_len=4096)


    def forward(self,x,past_k = None,past_v = None, use_cache = False):
        B, T, C = x.shape

        #Q
        q =self.q_proj(x)
        q = q.view(B,T,self.n_head,self.head_dim)
        q = q.transpose(1,2)
        # (B, n_head, T, head_dim)

        # K
        k = self.k_proj(x)
        k = k.view(B,T,self.n_kv_head,self.head_dim)
        k = k.transpose(1, 2)
        # (B, n_kv_head, T, head_dim)

        #V
        v = self.v_proj(x)
        v = v.view(B,T,self.n_kv_head,self.head_dim)
        v = v.transpose(1, 2)
        # (B, n_kv_head, T, head_dim)

        #RoPE
        if past_k is None:
            # Training / prefill.
            # Current tokens occupy:
            # 0, 1, 2, ..., T-1
            past_len = 0
        else:
            # During generation:
            # cached tokens already occupy:
            # 0 ... past_len-1
            past_len = past_k.size(2)

        positions = torch.arange(past_len,past_len + T,device=x.device)

        q = self.rope(q,positions)
        k = self.rope(k,positions)
        
        #KV Cache
        if past_k is not None:
            k = torch.cat([past_k, k],dim=2)

        if past_v is not None:
            v = torch.cat([past_v, v],dim=2)

        #Att
        if past_k is None:
            # Training / prefill
            # Q length == K/V length
            # Therefore normal causal attention.

            out = F.scaled_dot_product_attention(q,k,v,
                is_causal=True,
                dropout_p=(self.dropout_p if self.training else 0.0),
                enable_gqa=True
            )

        else:
            # Decode with KV cache
            # q = only the NEW token
            # k/v = entire cache
            # The new token should attend to ALL
            # cached tokens, so DO NOT use is_causal=True.

            out = F.scaled_dot_product_attention(q,k,v,
                is_causal=False,
                dropout_p=0.0,
                enable_gqa=True
            )

        out = out.transpose(1, 2)

        out = out.contiguous().view(
            B,
            T,
            C
        )

        out = self.proj(out)

        if use_cache:
            return out, k, v

        return out


In [43]:
class FeedForward(nn.Module):
    def __init__(self,n_embed) -> None:
        super().__init__()
        # Standard Transformer FFN:
        #
        # 4 * n_embed
        #
        # SwiGLU has two input projections, so we reduce
        # the hidden size to keep parameter count roughly
        # comparable to a normal 4x FFN.

        hidden_dim = int(
            (8 / 3) * n_embed
        )
        hidden_dim = (
            (hidden_dim + 7) // 8
        ) *  8
        self.w1 = nn.Linear(n_embed,hidden_dim,bias=False        )
        # x -> W2
        self.w2 = nn.Linear(n_embed,hidden_dim,bias=False)
        # output projection
        self.w3 = nn.Linear(hidden_dim,n_embed,bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # First branch:
        # Swish(xW1)
        gate = F.silu(self.w1(x))
        # Second branch:
        value = self.w2(x)
        # Elementwise gating:
        # Swish(xW1) * (xW2)
        x = gate * value
        # Output projection
        x = self.w3(x)
        return self.dropout(x)

In [44]:
class Block(nn.Module):
    def __init__(self,n_embed,n_head,n_kv_head) -> None:
        super().__init__()

        self.sa = MultiHeadAttention(
            n_embd=n_embed,
            n_head=n_head,
            n_kv_head=n_kv_head,
            dropout=dropout
        )

        self.ffwd = FeedForward(n_embed)
        self.norm1 = nn.RMSNorm(n_embed)
        self.norm2 = nn.RMSNorm(n_embed)

    def forward(self,x,past_k = None, past_v = None, use_cache = False):
        x_norm = self.norm1(x)

        if use_cache:
            attn_out,k,v = self.sa(
                x_norm,
                past_k = past_k,
                past_v = past_v,
                use_cache = True
            )
        else:
            attn_out = self.sa(x_norm)

        x = x + attn_out
        x = x + self.ffwd(self.norm2(x))

        if use_cache:
            return x, k, v

        return x

In [45]:
class Transformer(nn.Module):
    def __init__(self,n_embed,n_head,n_layer,n_kv_head) -> None:
        super().__init__()
        self.blocks = nn.ModuleList([
            Block(n_embed,n_head,n_kv_head) for _ in range(n_layer)
        ])
        self.token_embedding = nn.Embedding(vocab_size,n_embed)
        self.ln_f = nn.RMSNorm(n_embed)
        self.lm_head = nn.Linear(n_embed,vocab_size)

    def forward(self,x, targets = None, past_kvs = None, use_cache = False):
        B,T = x.shape
        x = self.token_embedding(x)

        # Transformer Blocks
        new_kvs = []

        for i, block in enumerate(self.blocks):
            if use_cache:
                past_k = None
                past_v = None

                if past_kvs is not None:
                    past_k, past_v = past_kvs[i]

                x,k,v = block(x, past_k = past_k, past_v = past_v, use_cache = True)
                new_kvs.append((k,v))
            else:
                x = block(x)
        
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B,T,C = logits.shape

            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B*T)

            loss = F.cross_entropy(logits_flat,targets_flat)
        if use_cache:
            return logits, loss, new_kvs

        return logits, loss

In [46]:
batch_size = 64
block_size = seq_len

n_embed = 128
n_head = 4
n_layer = 4
n_kv_head = 2

dropout = 0.1
learning_rate = 3e-4


warmup_steps = 500
max_steps = 10_000

min_lr = 3e-5
eval_interval = 500

assert n_embed % n_head == 0
assert n_head % n_kv_head == 0

head_dim = n_embed // n_head
assert head_dim % 2 == 0

In [47]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Transformer(
    n_embed=n_embed,
    n_head=n_head,
    n_layer=n_layer,
    n_kv_head=n_kv_head
).to(device)


In [48]:
def get_lr(step):

    # Warmup
    if step < warmup_steps:

        return learning_rate * (
            step + 1
        ) / warmup_steps

    # Cosine decay
    decay_ratio = (
        step - warmup_steps
    ) / (
        max_steps - warmup_steps
    )

    decay_ratio = min(
        decay_ratio,
        1.0
    )

    coeff = 0.5 * (
        1.0 + math.cos(
            math.pi * decay_ratio
        )
    )

    return min_lr + coeff * (
        learning_rate - min_lr
    )


In [49]:
num_params = sum(
    p.numel()
    for p in model.parameters()
    )

print(f"Parameters: {num_params:,}")

Parameters: 747,216


In [50]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.1,
    betas=(0.9, 0.95)
)

scaler = torch.amp.GradScaler("cuda")

In [51]:
@torch.no_grad()
def estimate_loss():

    model.eval()

    total_loss = 0.0
    total_batches = 0

    for xb, yb in val_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits, loss = model(
                xb,
                yb
            )

        total_loss += loss.item()
        total_batches += 1

    model.train()

    return total_loss / total_batches

In [52]:

global_step = 0

train_loss_accum = 0.0
train_loss_count = 0

model.train()

while global_step < max_steps:

    for xb, yb in train_loader:

        if global_step >= max_steps:
            break

        xb = xb.to(device)
        yb = yb.to(device)

        # ---------------------------
        # Learning rate
        # ---------------------------

        lr = get_lr(global_step)

        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        # ---------------------------
        # Forward
        # ---------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits, loss = model(
                xb,
                yb
            )

        # ---------------------------
        # Backward
        # ---------------------------

        scaler.scale(loss).backward()

        # Unscale gradients
        scaler.unscale_(optimizer)

        # Gradient clipping
        grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )
        )

        # ---------------------------
        # Optimizer step
        # ---------------------------

        scaler.step(optimizer)
        scaler.update()

        # ---------------------------
        # Track training loss
        # ---------------------------

        train_loss_accum += loss.item()
        train_loss_count += 1

        global_step += 1

        # ---------------------------
        # Validation
        # ---------------------------

        if global_step % eval_interval == 0:

            avg_train_loss = (
                train_loss_accum
                / train_loss_count
            )

            val_loss = estimate_loss()

            print(
                f"Step {global_step:5d} "
                f"| Train Loss: {avg_train_loss:.4f} "
                f"| Val Loss: {val_loss:.4f} "
                f"| LR: {lr:.2e} "
                f"| Grad Norm: {grad_norm:.2f}"
            )

            train_loss_accum = 0.0
            train_loss_count = 0

Step   500 | Train Loss: 2.8276 | Val Loss: 2.0110 | LR: 3.00e-04 | Grad Norm: 0.27
Step  1000 | Train Loss: 1.8383 | Val Loss: 1.6666 | LR: 2.98e-04 | Grad Norm: 0.31
Step  1500 | Train Loss: 1.6545 | Val Loss: 1.5564 | LR: 2.93e-04 | Grad Norm: 0.27
Step  2000 | Train Loss: 1.5733 | Val Loss: 1.4983 | LR: 2.84e-04 | Grad Norm: 0.30
Step  2500 | Train Loss: 1.5244 | Val Loss: 1.4609 | LR: 2.72e-04 | Grad Norm: 0.35
Step  3000 | Train Loss: 1.4903 | Val Loss: 1.4352 | LR: 2.56e-04 | Grad Norm: 0.31
Step  3500 | Train Loss: 1.4671 | Val Loss: 1.4180 | LR: 2.39e-04 | Grad Norm: 0.31
Step  4000 | Train Loss: 1.4493 | Val Loss: 1.4032 | LR: 2.19e-04 | Grad Norm: 0.29
Step  4500 | Train Loss: 1.4347 | Val Loss: 1.3920 | LR: 1.98e-04 | Grad Norm: 0.33
Step  5000 | Train Loss: 1.4228 | Val Loss: 1.3833 | LR: 1.76e-04 | Grad Norm: 0.31
Step  5500 | Train Loss: 1.4134 | Val Loss: 1.3772 | LR: 1.54e-04 | Grad Norm: 0.35
Step  6000 | Train Loss: 1.4062 | Val Loss: 1.3700 | LR: 1.32e-04 | Grad Nor

In [53]:
model.eval()

context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

past_kvs = None

with torch.no_grad():

    for step in range(256):

        if past_kvs is None:

            # First pass:
            # process the entire current context

            context_cond = context[:, -block_size:]

            logits, _, past_kvs = model(
                context_cond,
                use_cache=True
            )

        else:

            # Subsequent passes:
            # ONLY process the new token

            new_token = context[:, -1:]

            logits, _, past_kvs = model(
                new_token,
                past_kvs=past_kvs,
                use_cache=True
            )

        # Last token's logits

        logits = logits[:, -1, :]

        # Temperature can be added here
        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        context = torch.cat(
            (context, next_token),
            dim=1
        )

print(
    decode(
        context[0].tolist()
    )
)


Scotter. Now, daughter first the heavens more love. I have content I'll profit again. The going hours is more to you. A word; but he did follow where not. Exeunt [Sounds.] And what divide] Sir Hastings who shall yoke was the victory; their tarrying- and ev
